In [ ]:
from fastapi import FastAPI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

from langgraph.graph import StateGraph
from typing import TypedDict, List

# -----------------------------
# FASTAPI INIT
# -----------------------------
app = FastAPI()

# -----------------------------
# LLM + EMBEDDINGS
# -----------------------------
llm = ChatOpenAI(model="gpt-4o-mini")

embeddings = OpenAIEmbeddings()

# -----------------------------
# SAMPLE DOCUMENTS
# -----------------------------
docs = [
    Document(page_content="FastAPI is a modern web framework for APIs."),
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="LangGraph helps create multi-step workflows.")
]

# Create vector DB
vectorstore = FAISS.from_documents(docs, embeddings)

# -----------------------------
# LANGGRAPH STATE
# -----------------------------
class GraphState(TypedDict):
    question: str
    context: List[str]
    answer: str

# -----------------------------
# NODE 1: RETRIEVE DOCS
# -----------------------------
def retrieve(state: GraphState):
    query = state["question"]
    results = vectorstore.similarity_search(query, k=2)

    context = [doc.page_content for doc in results]

    return {"context": context}

# -----------------------------
# NODE 2: GENERATE ANSWER
# -----------------------------
def generate(state: GraphState):
    prompt = ChatPromptTemplate.from_template(
        "Answer the question using only the context.\n\nContext:\n{context}\n\nQuestion: {question}"
    )

    chain = prompt | llm

    response = chain.invoke({
        "context": "\n".join(state["context"]),
        "question": state["question"]
    })

    return {"answer": response.content}

# -----------------------------
# BUILD GRAPH
# -----------------------------
builder = StateGraph(GraphState)

builder.add_node("retrieve", retrieve)
builder.add_node("generate", generate)

builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "generate")

graph = builder.compile()

# -----------------------------
# FASTAPI ENDPOINT
# -----------------------------
@app.post("/ask")
async def ask_question(question: str):
    result = graph.invoke({"question": question})
    return {
        "question": question,
        "answer": result["answer"],
        "context_used": result["context"]
    }

ModuleNotFoundError: No module named 'langchain.vectorstores'